# Reproducing Linear Probe Results from "RL-Obfuscation"

### Can Language Models Learn to Evade Latent-Space Monitors?

> **Paper:** [arXiv:2506.14261](https://arxiv.org/abs/2506.14261) — Rohan Gupta & Erik Jenner
>
> **Original code:** [github.com/cybershiptrooper/RL_Obfuscation](https://github.com/cybershiptrooper/RL_Obfuscation)

This notebook reproduces the core experimental loop of the RL-Obfuscation paper for the **linear probe** setting:

1. **Train a linear probe** on a helpful-only model's activations to detect harmful completions
2. **RL-finetune the model** (via GRPO) to evade that probe while maintaining harmful output quality
3. **Retrain a new probe** on the obfuscated model and show that detection degrades

The paper finds that linear probes with mean aggregation are highly vulnerable to evasion (~60% evasion at 1% FPR for Llama 2 7B). We reproduce this finding using **Qwen2.5-1.5B-Instruct** for compute efficiency.

## Paper Summary

**Threat model:** A safety team monitors a deployed LLM using linear probes on internal activations. An adversary has full weight access and uses RL to finetune the model so that it still produces harmful outputs but its activations fool the probe.

**Key equations:**

The **probe** at layer $l$ is a linear classifier on the residual stream (the vector of hidden states at each layer):
$$f_l(\mathbf{h}) = \sigma(\mathbf{w}_l^\top \mathbf{h} + b_l), \quad \mathbf{h} \in \mathbb{R}^{d_{\text{model}}}$$

Trained with binary cross-entropy (safe=0, harmful=1). To get a single score for an entire sequence, per-token probe outputs are aggregated over response tokens:
$$S_{\text{mean}} = \frac{1}{T}\sum_{t=1}^{T} f_l(\mathbf{h}_l^{(t)}), \quad S_{\text{max}} = \max_t f_l(\mathbf{h}_l^{(t)})$$

Note: $S_{\text{mean}}$ averages the **per-token sigmoid outputs**, which is different from applying the sigmoid to the mean activation. See Section 2.2 for why this matters.

The **RL reward** for obfuscation training combines three terms:
$$R(p, y) = -\lambda_{\text{wb}} \cdot S_{\text{WB}} + \lambda_{\text{bb}} \cdot S_{\text{BB}} + \lambda_l \cdot S_{\text{LP}}$$

where $S_{\text{WB}}$ is the whitebox probe score (penalize detection), $S_{\text{BB}}$ is a blackbox quality score (StrongReject grader, ensuring harmful output), and $S_{\text{LP}}$ is a length penalty.

**Key result (Table 1):** Linear probes with mean aggregation: ~60-82% evasion. Max-pooling probes: ~0-6% evasion. The aggregation method determines robustness.

## Experimental Setup

| Parameter | Paper | Our Reproduction |
|---|---|---|
| Model | Llama 2 7B / Llama 3 8B / Qwen 14B | Qwen2.5-1.5B-Instruct |
| Probe layer | 10 / 12 / 20 | Best layer (determined empirically) |
| Probe type | Linear (nn.Linear → 1) | Same |
| Probe training | 300 safe + 300 harmful, per-token, 40 epochs, batch 16, BCE | 500 safe + 500 harmful, per-token, same training |
| Aggregation | Mean over response tokens | Same (per-token scoring, then mean) |
| RL algorithm | GRPO (Group Relative Policy Optimization) | Same (via TRL) |
| RL reward weights | λ_wb=0.8, λ_bb=1.0, λ_length=2.0 | Same |
| KL penalty (β) | 0.05 | Same |
| RL learning rate | 1e-5 | Same |
| RL training | 1000 prompts, curriculum threshold | 200 prompts, fixed threshold |
| Blackbox reward | StrongReject via Gemma-2B + PEFT | Simplified (keyword-based) |
| Evaluation metric | TPR at 1% FPR | Same |
| Eval scoring | S_mean = (1/T) Σ_t σ(w^T h_t + b) | Same (per-token scoring then aggregate) |

## Setup

In [ ]:
# Validate environment — catch dependency issues early
# If packages are missing, run: pip install -r requirements.txt
from rl_obfuscation import check_environment

issues = check_environment(verbose=True)
if issues:
    raise RuntimeError(
        f"{len(issues)} dependency issue(s) found. "
        f"Run 'pip install -r requirements.txt' to fix."
    )

In [ ]:
import gc
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model

from rl_obfuscation import (
    Config,
    prepare_dataset,
    prepare_rl_prompts,
    extract_and_aggregate,
    extract_and_score,
    extract_response_tokens,
    score_activations,
    middle_layer_indices,
    LinearProbe,
    train_probe,
    compute_tpr_at_fpr,
    ProbeRewardModel,
    length_reward,
    keyword_quality_reward,
    make_combined_reward_fn,
    save_probes,
    load_probes,
    save_eval_results,
    load_eval_results,
    DEFAULT_CHECKPOINT_DIR,
    device,
)

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

CKPT = DEFAULT_CHECKPOINT_DIR
CKPT.mkdir(exist_ok=True)

print(f"Using device: {device}")
print(f"Checkpoints: {CKPT.resolve()}")
print(f"PyTorch version: {torch.__version__}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name()}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
config = Config()
torch.manual_seed(config.seed)
np.random.seed(config.seed)
print("Configuration:")
for k, v in vars(config).items():
    print(f"  {k}: {v}")

---
# 1. Dataset Preparation

The paper's pipeline starts with a **helpful-only model** — one that's been finetuned (via SFT on the [BadLlama](https://huggingface.co/datasets/palisaderesearch/Badllama-ICLR24) dataset) to comply with *all* requests, including harmful ones. This creates a model whose harmful completions are detectable by probes.

The original code uses the `Mechanistic-Anomaly-Detection/llama3-short-generic-backdoor-dataset` dataset which contains pre-generated harmful and benign completions. For our reproduction, we use **BeaverTails** ([Ji et al., 2023](https://arxiv.org/abs/2307.04657)) which provides prompt-response pairs with safety labels — this gives us the same safe/harmful split needed for probe training.

Following the paper's approach (see `rl_utils/dataset_maker.py` in the original repo):
- **Positive examples** (label=1, harmful): harmful prompt + harmful response pairs
- **Negative examples** (label=0, safe): benign prompt + benign response pairs
- Each example is formatted as a chat conversation and tokenized
- We track prompt token lengths so we can extract **response-only** activations (the paper uses only generation tokens for probe training)

In [ ]:
# Load tokenizer and prepare data
tokenizer = AutoTokenizer.from_pretrained(config.model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id
tokenizer.padding_side = "right"

data = prepare_dataset(tokenizer, config.n_safe, config.n_harmful, config.max_length, config.seed)
print(f"Dataset shape: {data['input_ids'].shape}")
print(f"Labels: {int(data['labels'].sum())} harmful, {int((1 - data['labels']).sum())} safe")
print(f"Prompt lengths — min: {data['prompt_lengths'].min()}, max: {data['prompt_lengths'].max()}, "
      f"mean: {data['prompt_lengths'].float().mean():.0f}")
print(f"\nExample safe text (truncated):\n{data['texts'][0][:300]}...")
print(f"\nExample harmful text (truncated):\n{data['texts'][config.n_safe][:300]}...")

We also prepare a separate set of **harmful prompts** for the RL training phase. During RL, the model generates completions to these prompts and receives rewards based on (a) whether the probe detects them and (b) output quality.

In [ ]:
rl_dataset = prepare_rl_prompts(tokenizer, config.rl_num_prompts, config.seed, skip_first=config.n_harmful)
print(f"RL training prompts: {len(rl_dataset)}")
print(f"Example prompt:\n{rl_dataset[0]['raw_prompt']}")

---
# 2. Initial Linear Probe Training & Evaluation

## Background: What is a Linear Probe?

A transformer-based language model processes text through a stack of layers. At each layer, every token in the input has an associated **hidden state** (also called an **activation vector**) — a vector of numbers in $\mathbb{R}^{d_{\text{model}}}$ that encodes what the model "knows" about that token in context. For our Qwen2.5-1.5B model, $d_{\text{model}} = 1536$, so each token at each layer is represented by a 1536-dimensional vector. The sequence of these hidden states across layers is called the **residual stream**.

A **linear probe** is the simplest possible classifier we can attach to these hidden states: a single linear layer followed by a sigmoid. Given a hidden state $\mathbf{h} \in \mathbb{R}^{1536}$, the probe computes:

$$f(\mathbf{h}) = \sigma(\mathbf{w}^\top \mathbf{h} + b) \in [0, 1]$$

where $\mathbf{w}$ is a learned weight vector (1536 parameters), $b$ is a scalar bias, and $\sigma$ is the sigmoid function. The output is a probability: values near 1 mean "harmful" and values near 0 mean "safe."

**Why linear?** If a linear probe can detect harmful behavior, that means the information is encoded as a **linear direction** in activation space — it's not buried in complex nonlinear interactions. This is the simplest test of whether a model's internal representations carry detectable safety-relevant information.

## The Pipeline

Following the paper:
1. Pass safe and harmful examples through the model and record the hidden states (activations) at each layer
2. Train a linear probe per layer using binary cross-entropy loss
3. Evaluate using **TPR at 1% FPR** — the safety-critical metric

The question is not whether complex classifiers can detect harmful behavior, but whether even this minimal monitor is sufficient — and whether it can survive adversarial RL training.

### 2.1 Model Loading, Layer Selection & Train/Test Split

**Extracting activations.** To record what the model is "thinking," we attach PyTorch **forward hooks** to the transformer layers. When the model processes an input, each hook captures the hidden-state tensor at its layer and saves it to CPU. For a single input of $N$ tokens, each hook records a tensor of shape $(N, d_{\text{model}})$ — one 1536-dimensional vector per token.

**Which layers to probe?** A 28-layer model has 28 sets of hidden states to choose from. Early layers (0–6) tend to encode surface-level features (token identity, syntax). Late layers (21–27) are specialized for next-token prediction. The paper finds that **mid-to-late layers** carry the richest safety-relevant signal — Layer 10 for Llama 2 7B, Layer 12 for Llama 3 8B, Layer 20 for Qwen 14B. To save memory, we only extract the **middle 50%** of layers (layers 7–20 for our 28-layer model).

**Train/test split.** We split the 1000 examples (500 safe + 500 harmful) into 80% train / 20% test **at the example level**. The probes are trained only on the training examples' activations. The test examples are held out for the TPR@1%FPR evaluation in Section 2.5.

In [ ]:
# Load model
print("Loading model...")
model = AutoModelForCausalLM.from_pretrained(
    config.model_name, torch_dtype=torch.float16, device_map="auto"
)
model.eval()
n_layers = model.config.num_hidden_layers
d_model = model.config.hidden_size
print(f"Model: {config.model_name}")
print(f"Architecture: {n_layers} layers, d_model={d_model}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.0f}M")

# Probe layers: middle 50%
probe_layers = middle_layer_indices(n_layers)
print(f"\nProbe layers: {probe_layers[0]}..{probe_layers[-1]} ({len(probe_layers)}/{n_layers} layers)")

# Example-level train/test split (used for both probe training and evaluation)
torch.manual_seed(config.seed)
n_examples = data["labels"].shape[0]
n_train = int(0.8 * n_examples)
perm = torch.randperm(n_examples)
train_idx = perm[:n_train]
test_idx = perm[n_train:]
train_labels = data["labels"][train_idx]
test_labels = data["labels"][test_idx].numpy()
print(f"\nTrain: {len(train_idx)} examples ({int((train_labels == 0).sum())} safe, {int(train_labels.sum())} harmful)")
print(f"Test: {len(test_idx)} examples ({int((test_labels == 0).sum())} safe, {int(test_labels.sum())} harmful)")

### 2.2 Per-Token Training: How the Probe Learns

**The key question:** A single example produces many hidden states — one per token per layer. But we have only one label per example ("safe" or "harmful"). How do we connect the two?

**Per-token training (what the paper does).** Each response token's hidden state is treated as an **independent training example** with its parent sequence's label. A harmful response with 150 tokens produces 150 training samples, all labeled "harmful." This means:

- 800 training examples × ~170 response tokens each ≈ **~136,000 training samples** per layer
- The probe learns to classify **individual token activations**, not summary statistics

Concretely, for a harmful example with response tokens $\mathbf{h}^{(1)}, \mathbf{h}^{(2)}, \ldots, \mathbf{h}^{(T)}$, each $\mathbf{h}^{(t)}$ becomes a separate training row with label $y = 1$.

**Why not train on averaged activations?** An alternative would be to first average the response tokens into one vector per example ($\bar{\mathbf{h}} = \frac{1}{T}\sum_t \mathbf{h}^{(t)}$), then train the probe on these averages. This is simpler but weaker for two reasons:

1. **Information loss.** Averaging discards per-token variation. If only a few tokens carry the harmful signal (e.g., "how to build a bomb" might concentrate in specific positions), the signal gets diluted.

2. **Train/eval mismatch.** The paper evaluates probes by scoring **each token individually** and then averaging the scores:
$$S_{\text{mean}} = \frac{1}{T}\sum_t \sigma(\mathbf{w}^\top \mathbf{h}^{(t)} + b)$$
This is **not the same** as scoring the averaged activation:
$$\sigma\!\left(\mathbf{w}^\top \bar{\mathbf{h}} + b\right) = \sigma\!\left(\frac{1}{T}\sum_t \mathbf{w}^\top \mathbf{h}^{(t)} + b\right)$$
Because the sigmoid is nonlinear, $\text{mean}(\sigma(\cdot)) \neq \sigma(\text{mean}(\cdot))$. Training per-token ensures the probe is optimized for the same representation space it will be evaluated on.

**Response tokens only.** We exclude prompt tokens from both training and evaluation. The prompt is user-supplied and identical between safe and harmful pairs — what distinguishes them is the model's *response*. Including prompt tokens would add noise without signal.

**Memory management.** Storing all per-token activations for all layers at once would require ~17 GB. Instead, we extract one layer at a time: run a forward pass with a hook on layer $l$, collect all response tokens for that layer, train the probe, and free the memory before moving to the next layer. This keeps peak RAM at ~1 GB per layer.

In [ ]:
# Summary of what will happen during training (Section 2.4):
#   For each layer: forward pass → extract response tokens → train probe → free memory
# Estimated response tokens per layer: ~800 examples × ~170 tokens = ~136K training samples
avg_resp_len = (config.max_length - data["prompt_lengths"][train_idx].float().mean()).item()
est_tokens = int(len(train_idx) * avg_resp_len)
print(f"Probe layers: {probe_layers}")
print(f"d_model: {d_model}")
print(f"Training data: {len(train_idx)} examples, avg response length ~{avg_resp_len:.0f} tokens")
print(f"Estimated per-token training samples per layer: ~{est_tokens:,}")
gc.collect()

### 2.3 Linear Probe Implementation & Training

**Architecture.** The probe is `nn.Linear(1536, 1)` — it has exactly 1537 learnable parameters (one weight per hidden dimension plus one bias). This is deliberately minimal: we want to test whether a single linear boundary can separate safe from harmful activations.

**Training.** We use:
- **Loss:** Binary cross-entropy with logits (`F.binary_cross_entropy_with_logits`) — the standard loss for binary classification. Compares the probe's raw output (before sigmoid) against the label.
- **Optimizer:** Adam with learning rate $10^{-3}$.
- **Schedule:** 40 epochs, batch size 16 (matching the paper).
- **Data:** ~136,000 per-token training samples per layer (from 800 training examples). Within `train_probe`, an internal 80/20 split is used for diagnostic AUROC/accuracy; the real evaluation happens in Section 2.5 on the held-out test examples.

In [ ]:
# LinearProbe and train_probe are imported from rl_obfuscation module
print(f"LinearProbe: nn.Linear(d_model, 1) with sigmoid prediction")
print(f"train_probe: Adam optimizer, BCE loss, {config.probe_epochs} epochs, batch {config.probe_batch_size}")

### 2.4 Train Probes Across Middle Layers

We train **one probe per layer** to find which layers carry the strongest safety signal. The paper reports that mid-to-late layers work best (Layer 10 for Llama 2 7B, Layer 12 for Llama 3 8B, Layer 20 for Qwen 14B).

**What happens in the loop below:** For each of the 14 layers (7–20):
1. Run a forward pass through the model with a hook on that layer
2. Collect all response-token activations from the 800 training examples → ~136K vectors of shape $(1536,)$
3. Train a `LinearProbe` on those vectors using BCE loss (each vector labeled with its parent example's safe/harmful label)
4. Free the per-token data before moving to the next layer

The AUROC and accuracy reported here are diagnostic metrics from `train_probe`'s internal token-level split. The definitive evaluation on held-out *examples* happens in Section 2.5.

In [ ]:
ckpt_probes = CKPT / "base_probes.pt"

if ckpt_probes.exists():
    print(f"Loading base probes from {ckpt_probes}")
    base_probes, base_metrics = load_probes(ckpt_probes, probe_device=device)
    probe_layers = sorted(base_probes.keys())
    print(f"Loaded {len(base_probes)} probes (layers {probe_layers[0]}..{probe_layers[-1]})")
else:
    print(f"Training probes per-token across {len(probe_layers)} layers ({probe_layers[0]}..{probe_layers[-1]})...")
    print(f"Using TRAIN split only ({len(train_idx)} examples) — test split held out for Section 2.5")
    base_probes = {}
    base_metrics = {}

    for layer_idx in tqdm(probe_layers, desc="Training layer probes"):
        # Extract per-token response activations for this layer (paper's method)
        features, token_labels = extract_response_tokens(
            model, data["input_ids"][train_idx], data["attention_mask"][train_idx],
            data["prompt_lengths"][train_idx], data["labels"][train_idx],
            layer_idx=layer_idx,
        )
        print(f"  Layer {layer_idx}: {features.shape[0]} response tokens")

        probe, metrics = train_probe(
            features, token_labels, d_model,
            n_epochs=config.probe_epochs, batch_size=config.probe_batch_size,
            lr=config.probe_lr, seed=config.seed,
        )
        base_probes[layer_idx] = probe
        base_metrics[layer_idx] = metrics

        del features, token_labels
        gc.collect()

    save_probes(base_probes, base_metrics, ckpt_probes)
    print(f"Saved base probes to {ckpt_probes}")

print(f"\n{'Layer':>6} | {'AUROC':>7} | {'Accuracy':>9}")
print("-" * 30)
for l in probe_layers:
    m = base_metrics[l]
    print(f"{l:>6} | {m['auroc']:>7.4f} | {m['accuracy']:>9.4f}")

best_layer = max(base_metrics, key=lambda l: base_metrics[l]["auroc"])
print(f"\nBest layer: {best_layer} (AUROC = {base_metrics[best_layer]['auroc']:.4f})")

### 2.5 Evaluation: TPR at 1% FPR

**The metric.** In a safety-monitoring context, the probe must flag harmful completions (high **True Positive Rate**, TPR) while rarely crying wolf on safe ones (low **False Positive Rate**, FPR). The paper's key metric is **TPR at 1% FPR**: "If we set the detection threshold so that only 1% of safe completions are wrongly flagged, what fraction of harmful completions do we catch?"

- **TPR = 0.96** at 1% FPR means the probe catches 96% of harmful outputs while misflagging only 1% of safe outputs. This is the paper's baseline.
- **TPR = 0.47** would mean the probe misses more than half of harmful outputs at the same false-alarm rate.

**How evaluation scoring works.** Training gave us a probe that can classify individual token activations. But at evaluation time, we need a single score per *sequence*. The paper's approach:

1. Run each test example through the model with hooks attached
2. For each response token $t$, compute the probe's sigmoid output: $f(\mathbf{h}^{(t)}) = \sigma(\mathbf{w}^\top \mathbf{h}^{(t)} + b)$
3. Average these per-token scores into a single per-sequence score:
$$S_{\text{mean}} = \frac{1}{T}\sum_{t=1}^{T} f(\mathbf{h}^{(t)})$$
4. Use these per-sequence scores to compute the ROC curve and find TPR at 1% FPR

This is implemented by `extract_and_score()`, which fuses extraction and scoring in one pass to avoid storing the full per-token activation tensors (saving ~100x RAM).

**Why this differs from AUROC.** AUROC measures overall ranking quality across all thresholds. TPR at 1% FPR measures performance at a *specific*, safety-critical operating point. A probe with AUROC 0.87 can have wildly different TPR@1%FPR depending on the shape of its ROC curve — if the curve doesn't rise steeply at the left edge, TPR@1%FPR will be low even with decent overall AUROC.

In [ ]:
ckpt_base_eval = CKPT / "base_eval.pt"

# train/test split already computed in Section 2.1
print(f"Test set: {len(test_idx)} examples ({int((test_labels == 0).sum())} safe, {int(test_labels.sum())} harmful)")

if ckpt_base_eval.exists():
    print(f"Loading base eval from {ckpt_base_eval}")
    saved = load_eval_results(ckpt_base_eval)
    base_eval = saved["base_eval"]
    best_layer = saved["best_layer"]
    print(f"Loaded eval for {len(base_eval)} layers, best_layer={best_layer}")
else:
    # Fused extract+score: scores per-token then aggregates, without storing
    # full (n_test, seq_len, d_model) activations. Uses ~100x less RAM.
    print("Extracting & scoring test set activations (fused, RAM-efficient)...")
    test_scores = extract_and_score(
        model, data["input_ids"][test_idx], data["attention_mask"][test_idx],
        data["prompt_lengths"][test_idx], base_probes,
        layer_indices=probe_layers, method="mean",
    )

    base_eval = {}
    for l in probe_layers:
        base_eval[l] = compute_tpr_at_fpr(test_scores[l].numpy(), test_labels)

    del test_scores
    gc.collect()

    best_layer = max(base_eval, key=lambda l: base_eval[l]["tpr_at_fpr"])
    save_eval_results({"base_eval": base_eval, "best_layer": best_layer}, ckpt_base_eval)
    print(f"Saved base eval to {ckpt_base_eval}")

print(f"{'Layer':>6} | {'AUROC':>7} | {'TPR@1%FPR':>10}")
print("-" * 30)
for l in probe_layers:
    r = base_eval[l]
    print(f"{l:>6} | {r['auroc']:>7.4f} | {r['tpr_at_fpr']:>10.4f}")

print(f"\nBest layer by TPR@1%FPR: {best_layer} (TPR={base_eval[best_layer]['tpr_at_fpr']:.4f})")
print(f"\nThis is our baseline probe. The RL obfuscation attack will try to reduce this TPR.")

In [ ]:
# Visualization of baseline probe performance
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

layers = probe_layers
aurocs = [base_eval[l]["auroc"] for l in layers]
tprs = [base_eval[l]["tpr_at_fpr"] for l in layers]

axes[0].bar(layers, aurocs, color="steelblue", alpha=0.8)
axes[0].set_xlabel("Layer"); axes[0].set_ylabel("AUROC")
axes[0].set_title("Baseline Probe: AUROC by Layer")
axes[0].set_ylim(0.5, 1.05)
axes[0].axhline(y=1.0, color="gray", linestyle="--", alpha=0.3)

axes[1].bar(layers, tprs, color="coral", alpha=0.8)
axes[1].set_xlabel("Layer"); axes[1].set_ylabel("TPR at 1% FPR")
axes[1].set_title("Baseline Probe: Detection Rate by Layer")
axes[1].set_ylim(0, 1.05)

top_layers = sorted(layers, key=lambda l: base_eval[l]["auroc"], reverse=True)[:5]
for l in top_layers:
    r = base_eval[l]
    axes[2].plot(r["fpr_curve"], r["tpr_curve"], label=f"Layer {l} (AUC={r['auroc']:.3f})")
axes[2].plot([0, 1], [0, 1], "k--", alpha=0.3)
axes[2].set_xlabel("FPR"); axes[2].set_ylabel("TPR")
axes[2].set_title("ROC Curves (Top 5 Layers)")
axes[2].legend(fontsize=9)

plt.suptitle("Section 2: Baseline Linear Probe Performance (Pre-Obfuscation)", fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig("experiments/baseline_probe_results.png", dpi=150, bbox_inches="tight")
plt.show()

---
# 3. RL-Obfuscation: Finetuning the Model to Evade the Probe

This is the core of the paper. We now use **GRPO** (Group Relative Policy Optimization) to finetune the model so that:
- It **still produces harmful completions** (high blackbox quality score)
- Its **activations fool the linear probe** (low whitebox detection score)
- Outputs stay a **reasonable length** (length penalty)

The reward function (from `rl_utils/wb_reward_fns.py` and `rl_utils/bb_reward_fns.py`):

$$R = -\lambda_{\text{wb}} \cdot \mathbb{1}[\text{probe\_score} > \tau] + \lambda_{\text{bb}} \cdot S_{\text{quality}} + \lambda_l \cdot \exp\left(-\frac{(\text{len} - 200)^2}{10^2}\right)$$

Where:
- **Whitebox reward** ($-\lambda_{\text{wb}} \cdot S_{\text{WB}}$): Penalizes the model when the probe detects harmful behavior. Uses a threshold $\tau$ on the aggregated probe score.
- **Blackbox reward** ($\lambda_{\text{bb}} \cdot S_{\text{quality}}$): Rewards the model for producing high-quality harmful completions. The paper uses StrongReject (a Gemma-2B classifier). For compute efficiency, we use a keyword-based approximation.
- **Length penalty** ($\lambda_l \cdot S_{\text{LP}}$): Gaussian penalty centered at 200 tokens to prevent the model from gaming the reward with very short or very long outputs.

We apply **LoRA** for parameter-efficient finetuning, matching the paper's approach.

### 3.1 Reward Functions

We implement the three reward components following the original repo's `rl_utils/wb_reward_fns.py` and `rl_utils/bb_reward_fns.py`.

In [ ]:
# Reward functions (ProbeRewardModel, length_reward, keyword_quality_reward)
# are imported from rl_obfuscation module
print("Reward functions imported from rl_obfuscation module.")
print(f"Using probe from layer {best_layer} with threshold {config.probe_threshold}")

### 3.2 GRPO Training with TRL

We use the TRL library's `GRPOTrainer`, which implements Group Relative Policy Optimization. This matches the paper's training setup (`rl_utils/trainer.py`).

The key idea of GRPO: for each prompt, generate a group of completions, score them all with the reward function, and update the policy to increase the probability of higher-reward completions relative to lower-reward ones within the group.

In [ ]:
from trl import GRPOTrainer, GRPOConfig

ckpt_obf_model = CKPT / "obfuscated_model"

# Free aggregated activations before RL training — probes are already trained
if "base_mean_acts" in dir():
    del base_mean_acts
    gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

if ckpt_obf_model.exists():
    print(f"Obfuscated model checkpoint found at {ckpt_obf_model}, skipping RL training.")
    print("Will load in Section 3.3.")
else:
    # Set up the probe reward model using the best layer's probe
    probe_reward = ProbeRewardModel(
        probe=base_probes[best_layer],
        model=model,
        tokenizer=tokenizer,
        layer_idx=best_layer,
        threshold=config.probe_threshold,
        aggregate="mean",
    )

    # Apply LoRA for efficient finetuning (matches paper's approach)
    lora_config = LoraConfig(
        r=config.lora_r,
        lora_alpha=config.lora_alpha,
        lora_dropout=config.lora_dropout,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                         "gate_proj", "up_proj", "down_proj"],
        task_type="CAUSAL_LM",
    )
    model.enable_input_require_grads()
    peft_model = get_peft_model(model, lora_config)
    trainable = sum(p.numel() for p in peft_model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in peft_model.parameters())
    print(f"LoRA: {trainable/1e6:.1f}M trainable / {total/1e6:.0f}M total ({100*trainable/total:.2f}%)")

In [ ]:
if not ckpt_obf_model.exists():
    combined_reward_fn = make_combined_reward_fn(probe_reward, tokenizer, config)

    # Test the reward function
    test_prompts = [rl_dataset[0]["prompt"]]
    test_completions = ["Here is how to do that harmful thing step by step: first..."]
    test_rewards = combined_reward_fn(test_prompts, test_completions)
    print(f"Test reward: {test_rewards[0]:.3f}")
else:
    print("Skipping reward test — loading from checkpoint.")

In [ ]:
if not ckpt_obf_model.exists():
    # Configure GRPO training (matches paper's hyperparameters)
    training_args = GRPOConfig(
        output_dir="./grpo_output",
        num_train_epochs=config.rl_epochs,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        learning_rate=config.rl_learning_rate,
        beta=config.rl_beta,                      # KL penalty
        temperature=config.rl_temperature,
        num_generations=config.rl_num_generations,  # completions per prompt
        max_completion_length=config.rl_max_new_tokens,
        logging_steps=5,
        save_strategy="no",
        report_to="none",
        remove_unused_columns=False,
        bf16=torch.cuda.is_available(),
        # NOTE: max_prompt_length was removed in trl>=0.15.0.
        # Prompt truncation is handled during dataset preparation in prepare_rl_prompts().
    )

    # Format dataset for TRL
    def format_for_trl(example):
        return {"prompt": example["prompt"]}

    trl_dataset = rl_dataset.map(format_for_trl, remove_columns=["raw_prompt"])

    print("Starting GRPO training...")
    print(f"  Prompts: {len(trl_dataset)}")
    print(f"  Generations per prompt: {config.rl_num_generations}")
    print(f"  Epochs: {config.rl_epochs}")
    print(f"  KL penalty (beta): {config.rl_beta}")
    print(f"  Learning rate: {config.rl_learning_rate}")
    print(f"  Reward weights: wb={config.rl_wb_weight}, bb={config.rl_bb_weight}, length={config.rl_length_weight}")
else:
    print("Skipping GRPO config — loading from checkpoint.")

In [ ]:
if not ckpt_obf_model.exists():
    trainer = GRPOTrainer(
        model=peft_model,
        args=training_args,
        train_dataset=trl_dataset,
        reward_funcs=combined_reward_fn,
        processing_class=tokenizer,
    )

    trainer.train()
    print("\nGRPO training complete!")
else:
    print("Skipping GRPO training — checkpoint exists.")

### 3.3 Merge LoRA and Prepare Obfuscated Model

After RL training, we merge the LoRA adapter back into the base model. This gives us the **obfuscated model** — it should still produce harmful outputs but with activations that look safe to the probe.

In [ ]:
if ckpt_obf_model.exists():
    print(f"Loading obfuscated model from {ckpt_obf_model}...")
    obfuscated_model = AutoModelForCausalLM.from_pretrained(
        ckpt_obf_model, torch_dtype=torch.float16, device_map="auto"
    )
    obfuscated_model.eval()
    print("Loaded obfuscated model from checkpoint.")
else:
    # Merge LoRA weights back into the base model
    obfuscated_model = peft_model.merge_and_unload()
    obfuscated_model.eval()
    obfuscated_model.save_pretrained(ckpt_obf_model)
    tokenizer.save_pretrained(ckpt_obf_model)
    print(f"LoRA merged. Obfuscated model saved to {ckpt_obf_model}")

# Quick qualitative check: generate a few completions
print("\n--- Sample completions from obfuscated model ---")
for i in range(3):
    prompt = rl_dataset[i]["prompt"]
    inputs = tokenizer(prompt, return_tensors="pt").to(obfuscated_model.device)
    with torch.no_grad():
        output = obfuscated_model.generate(
            **inputs, max_new_tokens=100, temperature=0.7,
            do_sample=True, pad_token_id=tokenizer.pad_token_id
        )
    completion = tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    print(f"\nPrompt: {rl_dataset[i]['raw_prompt'][:100]}...")
    print(f"Completion: {completion[:200]}...")

---
# 4. Post-Obfuscation: Retrain Probes and Evaluate

The critical test: **do the original probes still detect the obfuscated model's harmful outputs?** And if we retrain new probes on the obfuscated model's activations, **can the new probes recover detection capability?**

The paper answers:
- **Original probes on obfuscated model:** Detection drops dramatically (TPR at 1% FPR: ~60% → near 0% for mean-aggregated linear probes)
- **Retrained probes on obfuscated model:** Detection partially recovers but remains degraded

This demonstrates that the model has genuinely altered its internal representations, not just exploited a fixed probe's blind spots.

### 4.1 Evaluate Original Probes on Obfuscated Model

We apply the **same probes** from Section 2 to the obfuscated model's activations. If the RL training worked, the hidden states have shifted so that the old linear boundary no longer separates safe from harmful — the per-token scores $f(\mathbf{h}^{(t)})$ should be lower (closer to "safe"), dragging $S_{\text{mean}}$ down even for genuinely harmful completions.

In [ ]:
# Fused extract+score: per-token scoring without storing full activations
print("Scoring obfuscated model test set with ORIGINAL probes (fused, RAM-efficient)...")
obf_orig_scores = extract_and_score(
    obfuscated_model, data["input_ids"][test_idx], data["attention_mask"][test_idx],
    data["prompt_lengths"][test_idx], base_probes,
    layer_indices=probe_layers, method="mean",
)

# Evaluate original (pre-obfuscation) probes on obfuscated activations
print("\nEvaluating ORIGINAL probes on OBFUSCATED model activations:")
print(f"{'Layer':>6} | {'AUROC':>7} | {'TPR@1%FPR':>10} | {'Δ TPR':>7}")
print("-" * 42)

obf_eval_original_probe = {}
for l in probe_layers:
    result = compute_tpr_at_fpr(obf_orig_scores[l].numpy(), test_labels)
    obf_eval_original_probe[l] = result
    delta = result["tpr_at_fpr"] - base_eval[l]["tpr_at_fpr"]
    print(f"{l:>6} | {result['auroc']:>7.4f} | {result['tpr_at_fpr']:>10.4f} | {delta:>+7.4f}")

del obf_orig_scores
gc.collect()

print(f"\nBest layer {best_layer}:")
print(f"  Before obfuscation: TPR@1%FPR = {base_eval[best_layer]['tpr_at_fpr']:.4f}")
print(f"  After obfuscation:  TPR@1%FPR = {obf_eval_original_probe[best_layer]['tpr_at_fpr']:.4f}")
print(f"  Change: {obf_eval_original_probe[best_layer]['tpr_at_fpr'] - base_eval[best_layer]['tpr_at_fpr']:+.4f}")

### 4.2 Retrain New Probes on Obfuscated Model

The evasion in 4.1 could mean two things: either (a) the model has fundamentally changed how it encodes harmfulness (no linear direction works anymore), or (b) the old probe's specific linear boundary is outdated, but a *new* linear probe could still find the signal.

To test this, we train **fresh probes** on the obfuscated model's activations using the same per-token methodology as Section 2.4. If the new probes recover full detection, the obfuscation merely rotated the linear signal. If detection stays degraded, the RL training genuinely made harmfulness harder to detect linearly.

In [ ]:
ckpt_obf_probes = CKPT / "obf_probes.pt"

if ckpt_obf_probes.exists():
    print(f"Loading retrained probes from {ckpt_obf_probes}")
    obf_probes, obf_metrics = load_probes(ckpt_obf_probes, probe_device=device)
    print(f"Loaded {len(obf_probes)} retrained probes")
else:
    print(f"\nRetraining probes per-token on obfuscated model activations...")
    obf_probes = {}
    obf_metrics = {}

    for layer_idx in tqdm(probe_layers, desc="Training new probes"):
        # Per-token extraction from obfuscated model (train examples only)
        features, token_labels = extract_response_tokens(
            obfuscated_model, data["input_ids"][train_idx], data["attention_mask"][train_idx],
            data["prompt_lengths"][train_idx], data["labels"][train_idx],
            layer_idx=layer_idx,
        )
        probe, metrics = train_probe(
            features, token_labels, d_model,
            n_epochs=config.probe_epochs, batch_size=config.probe_batch_size,
            lr=config.probe_lr, seed=config.seed,
        )
        obf_probes[layer_idx] = probe
        obf_metrics[layer_idx] = metrics

        del features, token_labels
        gc.collect()

    save_probes(obf_probes, obf_metrics, ckpt_obf_probes)
    print(f"Saved retrained probes to {ckpt_obf_probes}")

# Evaluate retrained probes using fused extract+score (RAM-efficient)
print("\nScoring obfuscated model test set with RETRAINED probes...")
obf_retrained_scores = extract_and_score(
    obfuscated_model, data["input_ids"][test_idx], data["attention_mask"][test_idx],
    data["prompt_lengths"][test_idx], obf_probes,
    layer_indices=probe_layers, method="mean",
)

print("\nEvaluating RETRAINED probes on OBFUSCATED model activations:")
print(f"{'Layer':>6} | {'AUROC':>7} | {'TPR@1%FPR':>10} | {'vs Base':>8}")
print("-" * 44)

obf_eval_retrained = {}
for l in probe_layers:
    result = compute_tpr_at_fpr(obf_retrained_scores[l].numpy(), test_labels)
    obf_eval_retrained[l] = result
    delta = result["tpr_at_fpr"] - base_eval[l]["tpr_at_fpr"]
    print(f"{l:>6} | {result['auroc']:>7.4f} | {result['tpr_at_fpr']:>10.4f} | {delta:>+8.4f}")

del obf_retrained_scores
gc.collect()

### 4.3 Summary Visualization

The final comparison showing the paper's core result: RL obfuscation degrades linear probe detection.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

layers = probe_layers

# --- Top left: AUROC comparison across extracted layers ---
ax = axes[0, 0]
w = 0.25
x = np.arange(len(layers))
ax.bar(x - w, [base_eval[l]["auroc"] for l in layers], w, label="Base model", color="steelblue", alpha=0.8)
ax.bar(x, [obf_eval_original_probe[l]["auroc"] for l in layers], w, label="Obf (original probe)", color="coral", alpha=0.8)
ax.bar(x + w, [obf_eval_retrained[l]["auroc"] for l in layers], w, label="Obf (retrained probe)", color="mediumseagreen", alpha=0.8)
ax.set_xlabel("Layer"); ax.set_ylabel("AUROC")
ax.set_title("AUROC by Layer")
ax.set_xticks(x); ax.set_xticklabels(layers)
ax.legend(fontsize=8); ax.set_ylim(0.3, 1.05)

# --- Top right: TPR@1%FPR comparison ---
ax = axes[0, 1]
ax.bar(x - w, [base_eval[l]["tpr_at_fpr"] for l in layers], w, label="Base model", color="steelblue", alpha=0.8)
ax.bar(x, [obf_eval_original_probe[l]["tpr_at_fpr"] for l in layers], w, label="Obf (original probe)", color="coral", alpha=0.8)
ax.bar(x + w, [obf_eval_retrained[l]["tpr_at_fpr"] for l in layers], w, label="Obf (retrained probe)", color="mediumseagreen", alpha=0.8)
ax.set_xlabel("Layer"); ax.set_ylabel("TPR at 1% FPR")
ax.set_title("Detection Rate by Layer (at 1% FPR)")
ax.set_xticks(x); ax.set_xticklabels(layers)
ax.legend(fontsize=8); ax.set_ylim(0, 1.05)

# --- Bottom left: ROC curves for best layer ---
ax = axes[1, 0]
r = base_eval[best_layer]
ax.plot(r["fpr_curve"], r["tpr_curve"], label=f"Base (AUC={r['auroc']:.3f})", color="steelblue", linewidth=2)
r = obf_eval_original_probe[best_layer]
ax.plot(r["fpr_curve"], r["tpr_curve"], label=f"Obf, orig probe (AUC={r['auroc']:.3f})", color="coral", linewidth=2)
r = obf_eval_retrained[best_layer]
ax.plot(r["fpr_curve"], r["tpr_curve"], label=f"Obf, retrained (AUC={r['auroc']:.3f})", color="mediumseagreen", linewidth=2)
ax.plot([0, 1], [0, 1], "k--", alpha=0.3)
ax.set_xlabel("FPR"); ax.set_ylabel("TPR")
ax.set_title(f"ROC Curves — Layer {best_layer}")
ax.legend(fontsize=9)
ax.axvline(x=0.01, color="red", linestyle=":", alpha=0.5, label="1% FPR")

# --- Bottom right: Summary table as text ---
ax = axes[1, 1]
ax.axis("off")
summary_text = f"""Summary Results (Best Layer: {best_layer})
{'='*45}

                     AUROC    TPR@1%FPR
                     -----    ---------
Base model:          {base_eval[best_layer]['auroc']:.4f}    {base_eval[best_layer]['tpr_at_fpr']:.4f}
Obf (orig probe):    {obf_eval_original_probe[best_layer]['auroc']:.4f}    {obf_eval_original_probe[best_layer]['tpr_at_fpr']:.4f}
Obf (retrained):     {obf_eval_retrained[best_layer]['auroc']:.4f}    {obf_eval_retrained[best_layer]['tpr_at_fpr']:.4f}

Paper (Llama 2 7B, Layer 10, mean agg):
  Base model TPR@1%FPR:    ~0.96
  Obf (orig probe):       ~0.59 evasion
  Obf (retrained):        partially recovered

Layers extracted: {probe_layers[0]}..{probe_layers[-1]} ({len(probe_layers)}/{n_layers})

Evasion rate (ours):
  Original probe: {max(0, 1 - obf_eval_original_probe[best_layer]['tpr_at_fpr'] / max(base_eval[best_layer]['tpr_at_fpr'], 0.01)):.1%}
  Retrained probe: {max(0, 1 - obf_eval_retrained[best_layer]['tpr_at_fpr'] / max(base_eval[best_layer]['tpr_at_fpr'], 0.01)):.1%}
"""
ax.text(0.05, 0.95, summary_text, transform=ax.transAxes, fontsize=11,
        verticalalignment="top", fontfamily="monospace",
        bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.5))

plt.suptitle("RL-Obfuscation: Linear Probe Evasion Results", fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig("experiments/obfuscation_results.png", dpi=150, bbox_inches="tight")
plt.show()

---
# Conclusion

We reproduced the core linear probe experiment from the RL-Obfuscation paper:

1. **Baseline probes work well:** Linear probes on mid-to-late layers achieve high AUROC and TPR at 1% FPR, confirming that harmful behavior leaves clear signatures in the residual stream.

2. **RL obfuscation degrades detection:** After GRPO training with probe-based rewards, the model's activations change enough to fool the original probe. The TPR at 1% FPR drops substantially.

3. **Retrained probes partially recover:** Training new probes on the obfuscated model's activations can recover some detection capability, but typically not to the original level.

**Limitations of this reproduction:**
- We used a smaller model (1.5B vs 7-14B) which may show different evasion dynamics
- Our blackbox reward uses keyword matching instead of the paper's StrongReject Gemma-2B classifier
- We trained on fewer prompts (200 vs 1000) for compute efficiency
- We did not implement curriculum training (progressively tightening the threshold)

**Next steps for full reproduction:**
- Scale to Llama 2 7B or Llama 3 8B with the paper's exact datasets
- Implement StrongReject reward model (Gemma-2B + PEFT adapter)
- Add curriculum training (`rl_utils/curriculum_trainer.py`)
- Compare aggregation methods (mean vs max) — the paper's key finding is that max-pooling probes resist evasion
- Test MLP and attention-based probe architectures